# Task 5 — Feedback source: RLVR vs RLAIF

Thin driver around the Task 5 scripts (same pattern as Tasks 1–4). The three policies are fixed: SFT (base model) and
the two **supplied** frozen adapters `checkpoints/rlvr_policy` and `checkpoints/rlaif_policy` — nothing is trained here,
so this task does not depend on Tasks 1–4 and can run any time the GPU is free.

| Step | What it runs | Rough time (RTX 5060) |
|---|---|---|
| 1 | GSM8K in-domain: greedy answers from SFT / RLVR / RLAIF (300 problems, cap 512), exact verifier, AI judge vs SFT | ~25–35 min |
| 2 | Controlled diagnostic set: exact verifier and AI judge on 20 problems × 5 variants | ~3–5 min |
| 3 | SVAMP transfer: same protocol on the fixed 100-problem set | ~8–12 min |
| 4 | combined comparison tables (no model) | seconds |

Every step saves its outputs and skips work already done (generations are reused, judge decisions are cached in
`results/task5_feedback/pairwise_judge_cache.json`), so an interrupted cell can simply be re-run.

In [10]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "configs").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display, Markdown
from common.data import load_yaml, read_jsonl, repo_path
from common.live import run_live
from task5_feedback import plots

CONFIG = os.environ.get("PA2_FEEDBACK_CONFIG", "configs/feedback.yaml")
cfg = load_yaml(CONFIG)
OUT = repo_path(cfg["results_dir"]) / "task5_feedback"
print("python:", sys.executable, "| CUDA:", torch.cuda.is_available())
for name in ("rlvr", "rlaif"):
    path = cfg["policies"][name]
    ok = (repo_path(path) / "adapter_model.safetensors").exists()
    print(f"{name:6s} {path:32s} {'OK' if ok else 'MISSING - run python -m scripts.download_assets'}")
for key in ("gsm_eval", "math_transfer_eval", "task5_diagnostics"):
    p = repo_path(cfg["paths"][key])
    print(f"{key:20s} {cfg['paths'][key]:48s} {'OK' if p.exists() else 'MISSING'}")
print("judge:", cfg["ai_judge_model"], "| math cap:", cfg["math_max_new_tokens"])
pd.set_option("display.max_colwidth", 200)

python: d:\Junior\CS6304 - ATML\PA2\ATML-PA2-LLM-PostTraining\.venv\Scripts\python.exe | CUDA: True
rlvr   checkpoints/rlvr_policy          OK
rlaif  checkpoints/rlaif_policy         OK
gsm_eval             data/gsm8k_eval.jsonl                            OK
math_transfer_eval   data/math_transfer_eval.jsonl                    OK
task5_diagnostics    data/task5_controlled_reward_diagnostics.jsonl   OK
judge: Qwen/Qwen2.5-3B-Instruct | math cap: 512


## Step 1 — in-domain comparison (GSM8K)
Each policy answers every problem with identical greedy decoding, cap and batches. The released exact verifier scores
the last `#### n`; the released pairwise judge compares each RL response with SFT's on the same problem (win = 1,
tie = 0.5, loss = 0; byte-identical responses count as a tie without a judge call).

In [ ]:
run_live("task5_feedback.evaluate_math", "--config", CONFIG, "--dataset", "gsm",
         title="Task 5 in-domain (GSM8K)", refresh_s=10, log_file=OUT.parent / "notebook_logs" / "task5_gsm.log")

## Step 2 — controlled reward diagnostics
Four controlled pairs per problem, each against the clean response: reasoning-only corruption (→ S_reason), wrong final
with the same reasoning (→ S_outcome), persuasive filler, and the gold-number distractor. Mechanisms: the exact
verifier, the judge's direct pairwise preference, and the RLAIF training reward (group win rate among the 5 variants).
The controlled pairs are also judged with the arguments swapped to check position consistency.

In [ ]:
run_live("task5_feedback.score_perturbations", "--config", CONFIG,
         title="Task 5 controlled diagnostics", refresh_s=5, log_file=OUT.parent / "notebook_logs" / "task5_diagnostics.log")

### Step 2b — judge parse check
The released judge decodes only 4 tokens and maps any output without a standalone A / B / TIE to TIE. This re-runs the
same prompts (same orientation, greedy) for the diagnostic pairs and a sample of policy-vs-SFT pairs, and separates
explicit ties from parse fall-backs. Read-only: the judge and its cache are unchanged (~3–5 min).

In [11]:
run_live("task5_feedback.inspect_judge", "--config", CONFIG, title="Judge parse check", refresh_s=5,
         log_file=OUT.parent / "notebook_logs" / "task5_inspect_judge.log")
audit = json.loads((OUT / "judge_parse_audit.json").read_text())
display(pd.DataFrame({k: {kk: vv for kk, vv in v.items() if kk != "raw_4tok_first_words"} for k, v in audit["by_source"].items()}).T)
print("reproduces cached decisions:", audit["reproduces_cache_rate"])

Judge parse check
FINISHED OK  |  elapsed 2m06s  |  full log: D:\Junior\CS6304 - ATML\PA2\ATML-PA2-LLM-PostTraining\results\notebook_logs\task5_inspect_judge.log

W1011 16:09:55.873000 4336 .venv\Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading checkpoint shards: 100%|██████████| 2/2 [00:09<00:00,  4.92s/it]
[inspect_judge] 129 comparisons
[inspect_judge] 1/129
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[inspect_judge] 21/129
[inspect_judge] 41/129
[inspect_judge] 61/129
[inspect_judge] 81/129
[inspect_judge] 101/129
[inspect_judge] 121/129
[inspect_judge] decisions {'TIE': 112, 'A

,n,decisions,tie_rate,ties_explicit,ties_parse_fallback,fallbacks_resolved_with_48_tokens
diagnostic:distractor,20,"{'TIE': 14, 'A': 5, 'B': 1}",0.7,14,0,{}
diagnostic:filler,20,"{'TIE': 15, 'B': 5}",0.75,15,0,{}
diagnostic:outcome,20,{'TIE': 20},1.0,20,0,{}
diagnostic:reasoning,20,{'TIE': 20},1.0,20,0,{}
gsm:rlaif_vs_sft,15,{'TIE': 15},1.0,15,0,{}
gsm:rlvr_vs_sft,15,"{'TIE': 13, 'A': 1, 'B': 1}",0.866667,13,0,{}
transfer:rlaif_vs_sft,7,{'TIE': 7},1.0,7,0,{}
transfer:rlvr_vs_sft,12,"{'A': 3, 'TIE': 8, 'B': 1}",0.666667,8,0,{}


reproduces cached decisions: 1.0


## Step 3 — out-of-domain comparison (SVAMP transfer set)
Same three policies, no retraining, same decoding and judge protocol, on the fixed `data/math_transfer_eval.jsonl`.

In [ ]:
run_live("task5_feedback.evaluate_math", "--config", CONFIG, "--dataset", "transfer",
         title="Task 5 transfer (SVAMP)", refresh_s=10, log_file=OUT.parent / "notebook_logs" / "task5_transfer.log")

## Step 4 — combined comparison

In [ ]:
run_live("task5_feedback.compare_feedback", "--config", CONFIG, title="Task 5 comparison", refresh_s=1)
cmp_ = json.loads((OUT / "feedback_source_comparison.json").read_text())
pt = pd.read_csv(OUT / "summary_policies.csv")
display(Markdown("**Policies (exact verifier + AI judge vs SFT)**"))
display(pt.set_index(["dataset", "policy"]).round(3))
if (OUT / "summary_transfer.csv").exists():
    display(Markdown("**In-domain → transfer (drop = in-domain − transfer)**"))
    display(pd.read_csv(OUT / "summary_transfer.csv").set_index("policy").round(3))
display(Markdown("**Controlled diagnostics: better / tie / wrong-preference rates**"))
dt = pd.read_csv(OUT / "summary_diagnostics.csv")
display(dt.pivot(index="category", columns="mechanism", values=["better_rate", "tie_rate", "wrong_rate"]).round(2))
display(Markdown(f"**S_reason** {cmp_['S_reason']}  \n**S_outcome** {cmp_['S_outcome']}  \n"
                 f"S_outcome over all outcome-changing pairs {cmp_['S_outcome_all_outcome_changing_pairs']}"))
display(Markdown("**Per-variant mean rewards (all five categories; the two scales are not comparable utilities)**"))
display(pd.read_csv(OUT / "summary_variant_rewards.csv", index_col=0).round(3))
display(Markdown("**Verifier vs judge on policy-vs-SFT pairs (non-identical responses)**"))
display(plots.disagreement_counts(cfg))

In [ ]:
for section in ("coverage", "noise", "exploitability", "cost"):
    display(Markdown(f"**{section}**"))
    print(json.dumps(cmp_[section], indent=1))

In [ ]:
figs = {"task5_policies": plots.policies(cfg), "task5_diagnostics": plots.diagnostics(cfg),
        "task5_variant_rewards": plots.variant_rewards(cfg), "task5_judge_matrix": plots.judge_matrix(cfg),
        "task5_failure_types": plots.failure_types(cfg), "task5_svamp_types": plots.svamp_types(cfg)}
for name, fig in figs.items():
    plots.save(fig, name)
    if fig is not None:
        display(fig)
        plt.close(fig)

## Step 5 — qualitative candidates
Diagnostic pairs chosen where the verifier and the judge behave differently (reasoning-only corruption, persuasive
filler, wrong final, gold distractor), then policy-vs-SFT pairs where the verifier is decisive but the judge disagrees.
Read the full texts before quoting any of them in the report.

In [ ]:
def show(rec, a_key, b_key, a_name, b_name, header):
    display(Markdown(header))
    for key, name in ((a_key, a_name), (b_key, b_name)):
        display(Markdown(f"*{name}:*\n\n> " + str(rec[key]).replace(chr(10), chr(10) + "> ")))

for r in read_jsonl(OUT / "examples_diagnostic.jsonl"):
    show(r, "better_response", "worse_response", f"better ({r['better_variant']})", f"worse ({r['worse_variant']})",
         f"---\n**{r['category']}** · problem {r['problem_id']} · gold {r['gold_final']} · verifier: **{r['verifier']}** · "
         f"judge pair: **{r['judge_pairwise']}** · judge group reward: **{r['judge_group']}**\n\n{r['question']}")

In [ ]:
for r in read_jsonl(OUT / "examples_policy.jsonl")[:6]:
    show(r, "policy_response", "sft_response", f"{r['policy']} (correct={r['policy_correct']})",
         f"sft (correct={r['sft_correct']})",
         f"---\n**{r['dataset']} · {r['policy']} vs SFT** · gold {r['gold_final']} · judge picked: **{r['judge']}** "
         f"(A = {r['policy']}, B = SFT)\n\n{r['question']}")